<a href="https://colab.research.google.com/github/Exper626/irpWhiteTea/blob/main/ann_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install ultralytics scikit-learn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.7 MB/s eta 0:00:00


In [3]:
import os
import cv2
import numpy as np
import json
import torch
from ultralytics import YOLO
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import pickle

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Config

In [4]:
# Best performing model from training experiments
# YOLOv8n won on mAP50 — use this for feature extraction
BEST_MODEL    = '/content/drive/MyDrive/yolo_weights/yolov8n_whitetea_best.pt'
DATASET_DIR   = '/content/drive/MyDrive/augment_herman'
OUTPUT_DIR    = '/content/drive/MyDrive/ann_pipeline'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# All image splits to extract features from
SPLITS = ['train', 'valid', 'test']

Load YOLO and extract detection features

In [5]:
model = YOLO(BEST_MODEL)

def extract_features(image_path, conf_threshold=0.25):
    """
    Run YOLO inference on one image and extract
    detection-derived features for ANN input.

    Features:
        bud_count       — total detections in image
        mean_confidence — average confidence of detections
        mean_box_area   — average normalised bounding box area
        density         — buds per unit of image area (proxy)
        spatial_spread  — std of bud x-centre positions (distribution)
        centre_bias     — mean distance of buds from image centre
    """
    results = model.predict(
        source=image_path,
        conf=conf_threshold,
        verbose=False
    )[0]

    boxes = results.boxes

    if boxes is None or len(boxes) == 0:
        return {
            'bud_count':       0,
            'mean_confidence': 0.0,
            'mean_box_area':   0.0,
            'density':         0.0,
            'spatial_spread':  0.0,
            'centre_bias':     0.0,
        }

    # Normalised xywh
    xywhn = boxes.xywhn.cpu().numpy()   # shape (N, 4): cx, cy, w, h
    confs = boxes.conf.cpu().numpy()     # shape (N,)

    cx = xywhn[:, 0]
    cy = xywhn[:, 1]
    bw = xywhn[:, 2]
    bh = xywhn[:, 3]

    areas = bw * bh
    count = len(boxes)

    # Distance from image centre (0.5, 0.5)
    dist_from_centre = np.sqrt((cx - 0.5)**2 + (cy - 0.5)**2)

    return {
        'bud_count':       count,
        'mean_confidence': float(np.mean(confs)),
        'mean_box_area':   float(np.mean(areas)),
        'density':         float(count / 1.0),        # normalised image area = 1
        'spatial_spread':  float(np.std(cx)) if count > 1 else 0.0,
        'centre_bias':     float(np.mean(dist_from_centre)),
    }

print('Feature extractor ready.')

Feature extractor ready.


Extract features from all images

In [6]:
all_features = []
all_image_paths = []

for split in SPLITS:
    img_dir = os.path.join(DATASET_DIR, split, 'images')
    if not os.path.exists(img_dir):
        print(f'Skipping {split} — folder not found')
        continue

    images = [f for f in os.listdir(img_dir) if f.endswith('.jpg')]
    print(f'\nExtracting features from {split}: {len(images)} images')

    for img_file in images:
        img_path = os.path.join(img_dir, img_file)
        features = extract_features(img_path)
        features['split']      = split
        features['image_file'] = img_file
        all_features.append(features)
        all_image_paths.append(img_path)

print(f'\nTotal feature vectors extracted: {len(all_features)}')

# Save features to JSON for inspection and reproducibility
features_path = os.path.join(OUTPUT_DIR, 'extracted_features.json')
with open(features_path, 'w') as f:
    json.dump(all_features, f, indent=2)

print(f'Saved to: {features_path}')


Extracting features from train: 342 images

Extracting features from valid: 7 images

Extracting features from test: 8 images

Total feature vectors extracted: 357
Saved to: /content/drive/MyDrive/ann_pipeline/extracted_features.json


Print feature summary

In [7]:
import pandas as pd

df = pd.DataFrame(all_features)
print(df[['bud_count','mean_confidence','mean_box_area',
          'density','spatial_spread','centre_bias']].describe())
print(f'\nImages with zero detections: {(df["bud_count"]==0).sum()}')
print(f'Images with detections:      {(df["bud_count"]>0).sum()}')
print(f'\nSplit breakdown:')
print(df['split'].value_counts())

        bud_count  mean_confidence  mean_box_area     density  spatial_spread  \
count  357.000000       357.000000     357.000000  357.000000      357.000000   
mean     1.414566         0.281648       0.001029    1.414566        0.055240   
std      1.973593         0.254773       0.001598    1.973593        0.102023   
min      0.000000         0.000000       0.000000    0.000000        0.000000   
25%      0.000000         0.000000       0.000000    0.000000        0.000000   
50%      1.000000         0.315272       0.000498    1.000000        0.000000   
75%      2.000000         0.485143       0.001155    2.000000        0.064754   
max     12.000000         0.966870       0.007946   12.000000        0.454556   

       centre_bias  
count   357.000000  
mean      0.168533  
std       0.165642  
min       0.000000  
25%       0.000000  
50%       0.169165  
75%       0.291433  
max       0.536278  

Images with zero detections: 141
Images with detections:      216

Split breakdo

ANN architecture (feasibility demonstration)

In [8]:
# ── IMPORTANT NOTE ─────────────────────────────────────────────────────────
# Ground-truth harvest readiness labels (ready/not_ready) were not available
# within the scope of this project, as they require longitudinal field
# assessment data correlating visual canopy state with expert harvest decisions.
#
# This ANN is implemented as a feasibility architecture and pipeline
# demonstration. The feature extraction → classification pipeline is fully
# operational. Empirical validation requires future data collection.
# ───────────────────────────────────────────────────────────────────────────

FEATURE_COLS = [
    'bud_count',
    'mean_confidence',
    'mean_box_area',
    'density',
    'spatial_spread',
    'centre_bias'
]

# Extract feature matrix
X = df[FEATURE_COLS].values

# ── Simulate readiness labels using threshold rule ──────────────────────────
# This is a DEMONSTRATION ONLY label — not validated ground truth.
# Threshold: images with >= 3 detected buds are flagged as 'approaching ready'
# This mirrors the field manager's description of harvest density indicators.
# Real labels require expert annotation of each image.

DETECTION_THRESHOLD = 3

y_demo = np.where(df['bud_count'] >= DETECTION_THRESHOLD, 1, 0)
label_names = ['not_ready', 'approaching_ready']

print('Class distribution in demonstration labels:')
print(f'  not_ready (0):         {(y_demo==0).sum()}')
print(f'  approaching_ready (1): {(y_demo==1).sum()}')
print('\nNOTE: These are threshold-derived demonstration labels.')
print('Ground-truth validation requires longitudinal expert assessment.')

Class distribution in demonstration labels:
  not_ready (0):         292
  approaching_ready (1): 65

NOTE: These are threshold-derived demonstration labels.
Ground-truth validation requires longitudinal expert assessment.


Train ANN

In [9]:
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_demo,
    test_size=0.2,
    random_state=42,
    stratify=y_demo if len(np.unique(y_demo)) > 1 else None
)

# ANN architecture
# Input: 6 detection-derived features
# Hidden: two layers of 32 and 16 neurons
# Output: binary classification (not_ready / approaching_ready)
ann = MLPClassifier(
    hidden_layer_sizes=(32, 16),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.2,
    verbose=False
)

ann.fit(X_train, y_train)

print('ANN training complete.')
print(f'Training accuracy:   {ann.score(X_train, y_train):.4f}')
print(f'Validation accuracy: {ann.score(X_test,  y_test):.4f}')
print()
print('Classification Report (DEMONSTRATION LABELS):')
print(classification_report(y_test, ann.predict(X_test),
                             target_names=label_names))
print()
print('Confusion Matrix:')
print(confusion_matrix(y_test, ann.predict(X_test)))

ANN training complete.
Training accuracy:   0.9018
Validation accuracy: 0.9444

Classification Report (DEMONSTRATION LABELS):
                   precision    recall  f1-score   support

        not_ready       0.94      1.00      0.97        59
approaching_ready       1.00      0.69      0.82        13

         accuracy                           0.94        72
        macro avg       0.97      0.85      0.89        72
     weighted avg       0.95      0.94      0.94        72


Confusion Matrix:
[[59  0]
 [ 4  9]]


Save ANN and scaler

In [10]:
ann_path    = os.path.join(OUTPUT_DIR, 'ann_model.pkl')
scaler_path = os.path.join(OUTPUT_DIR, 'scaler.pkl')

with open(ann_path,    'wb') as f: pickle.dump(ann,    f)
with open(scaler_path, 'wb') as f: pickle.dump(scaler, f)

print(f'ANN saved:    {ann_path}')
print(f'Scaler saved: {scaler_path}')

ANN saved:    /content/drive/MyDrive/ann_pipeline/ann_model.pkl
Scaler saved: /content/drive/MyDrive/ann_pipeline/scaler.pkl
